# Stage 1 — Data Load & Sanity Checks

Loads the 8 Complete Journey tables via `completejourney_py`, saves them as parquet for
fast reuse in later stages, and runs sanity checks on household counts, campaign targeting,
and redemption behavior before any feature engineering happens.


In [1]:
import pandas as pd
from pathlib import Path
import completejourney_py as cj

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load all 8 tables

In [2]:
data = cj.get_data()

for name, df in data.items():
    print(f"{name:<22} {df.shape[0]:>10,} rows  x {df.shape[1]} cols")


campaign_descriptions          27 rows  x 4 cols
coupons                   116,204 rows  x 3 cols
promotions             20,940,529 rows  x 5 cols
campaigns                   6,589 rows  x 2 cols
demographics                  801 rows  x 8 cols
transactions            1,469,307 rows  x 11 cols
coupon_redemptions          2,102 rows  x 4 cols
products                   92,331 rows  x 7 cols


## 2. Save as parquet

Parquet keeps dtypes (dates, ints) intact and loads much faster than re-pulling from
`completejourney_py` in every later notebook.


In [3]:
for name, df in data.items():
    df.to_parquet(RAW_DIR / f"{name}.parquet", index=False)

sorted(p.name for p in RAW_DIR.glob("*.parquet"))


['campaign_descriptions.parquet',
 'campaigns.parquet',
 'coupon_redemptions.parquet',
 'coupons.parquet',
 'demographics.parquet',
 'products.parquet',
 'promotions.parquet',
 'transactions.parquet']

## 3. Schema and preview of each table

In [4]:
for name, df in data.items():
    print("=" * 80)
    print(name, df.shape)
    print(df.dtypes)
    display(df.head())


campaign_descriptions (27, 4)
campaign_id               int64
campaign_type               str
start_date       datetime64[ns]
end_date         datetime64[ns]
dtype: object


,campaign_id,campaign_type,start_date,end_date
0,1,Type B,2017-03-03,2017-04-09
1,2,Type B,2017-03-08,2017-04-09
2,3,Type C,2017-03-13,2017-05-08
3,4,Type B,2017-03-29,2017-04-30
4,5,Type B,2017-04-03,2017-05-07


coupons (116204, 3)
coupon_upc     int64
product_id     int64
campaign_id    int64
dtype: object


,coupon_upc,product_id,campaign_id
0,10000085207,9676830,26
1,10000085207,9676943,26
2,10000085207,9676944,26
3,10000085207,9676947,26
4,10000085207,9677008,26


promotions (20940529, 5)
product_id          int64
store_id            int64
display_location      str
mailer_location       str
week                int64
dtype: object


...

,household_id,store_id,basket_id,product_id,quantity,sales_value,retail_disc,coupon_disc,coupon_match_disc,week,transaction_timestamp
0,900,330,31198570044,1095275,1,0.50,0.00,0.0,0.0,1,2017-01-01 11:53:26
1,900,330,31198570047,9878513,1,0.99,0.10,0.0,0.0,1,2017-01-01 12:10:28
2,1228,406,31198655051,1041453,1,1.43,0.15,0.0,0.0,1,2017-01-01 12:26:30
3,906,319,31198705046,1020156,1,1.50,0.29,0.0,0.0,1,2017-01-01 12:30:27
4,906,319,31198705046,1053875,2,2.78,0.80,0.0,0.0,1,2017-01-01 12:30:27


coupon_redemptions (2102, 4)
household_id                int64
coupon_upc                  int64
campaign_id                 int64
redemption_date    datetime64[ns]
dtype: object


,household_id,coupon_upc,campaign_id,redemption_date
0,1029,51380041013,26,2017-01-01
1,1029,51380041313,26,2017-01-01
2,165,53377610033,26,2017-01-03
3,712,51380041013,26,2017-01-07
4,712,54300016033,26,2017-01-07


products (92331, 7)
product_id          int64
manufacturer_id     int64
department            str
brand                 str
product_category      str
product_type          str
package_size          str
dtype: object


,product_id,manufacturer_id,department,brand,product_category,product_type,package_size
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISCELLANEOUS,National,NaN,NaN,NaN
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,NaN
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


## 4. Sanity checks

### 4.1 Household count


In [5]:
household_count = data["transactions"]["household_id"].nunique()
print(f"Unique households with transactions: {household_count:,}")


Unique households with transactions: 2,469


### 4.2 Week range

In [6]:
week_min = data["transactions"]["week"].min()
week_max = data["transactions"]["week"].max()
print(f"Week range: {week_min} to {week_max} ({week_max - week_min + 1} weeks)")


Week range: 1 to 53 (53 weeks)


### 4.3 Total sales

In [7]:
total_sales = data["transactions"]["sales_value"].sum()
print(f"Total sales value: ${total_sales:,.2f}")


Total sales value: $4,596,039.58


### 4.4 Demographics count

Demographics are self-reported (a voluntary survey), so coverage is expected to be far
below the full household base — this matters later: any model feature built from
demographics will be missing for most households.


In [8]:
demographics_count = data["demographics"]["household_id"].nunique()
print(f"Households with demographics: {demographics_count:,} "
      f"({demographics_count / household_count:.1%} of all households)")


Households with demographics: 801 (32.4% of all households)


### 4.5 Campaign summary table

In [9]:
targeted = (
    data["campaigns"].groupby("campaign_id")["household_id"].nunique().rename("targeted_households")
)
redeeming = (
    data["coupon_redemptions"].groupby("campaign_id")["household_id"].nunique().rename("redeeming_households")
)

campaign_summary = (
    data["campaign_descriptions"]
    .set_index("campaign_id")
    .join(targeted)
    .join(redeeming)
    .fillna(0)
)
campaign_summary["redeeming_households"] = campaign_summary["redeeming_households"].astype(int)
campaign_summary["redemption_rate"] = (
    campaign_summary["redeeming_households"] / campaign_summary["targeted_households"]
).round(4)

campaign_summary = campaign_summary.sort_index()
campaign_summary


,campaign_type,start_date,end_date,targeted_households,redeeming_households,redemption_rate
campaign_id,,,,,,
1,Type B,2017-03-03,2017-04-09,13,1,0.0769
2,Type B,2017-03-08,2017-04-09,48,2,0.0417
3,Type C,2017-03-13,2017-05-08,12,2,0.1667
4,Type B,2017-03-29,2017-04-30,81,6,0.0741
5,Type B,2017-04-03,2017-05-07,166,8,0.0482
6,Type C,2017-04-19,2017-05-21,65,1,0.0154
7,Type B,2017-04-24,2017-05-28,198,5,0.0253
8,Type A,2017-05-08,2017-06-25,1076,158,0.1468
9,Type B,2017-05-31,2017-07-02,176,20,0.1136


### 4.6 Redemption → targeting consistency check

Every household that redeemed a coupon for a campaign should also appear in that
campaign's targeted household list — a redemption with no matching target row would
indicate a data error or a leak in how campaigns/redemptions are linked.


In [10]:
redemption_pairs = data["coupon_redemptions"][["household_id", "campaign_id"]].drop_duplicates()

check = redemption_pairs.merge(
    data["campaigns"], on=["household_id", "campaign_id"], how="left", indicator=True
)
unmatched = check[check["_merge"] == "left_only"]

print(f"Redemption household/campaign pairs checked: {len(check):,}")
print(f"Pairs with no matching targeted household:   {len(unmatched):,}")
assert len(unmatched) == 0, "Found redemptions from households that were never targeted"


Redemption household/campaign pairs checked: 792
Pairs with no matching targeted household:   0


### 4.7 Treated vs. control counts for campaign 18

In [11]:
all_households = set(data["transactions"]["household_id"].unique())
treated_18 = set(data["campaigns"].loc[data["campaigns"]["campaign_id"] == 18, "household_id"])
control_18 = all_households - treated_18

print(f"Campaign 18 — treated (targeted): {len(treated_18):,}")
print(f"Campaign 18 — control (untargeted): {len(control_18):,}")
print(f"Total households: {len(all_households):,}")


Campaign 18 — treated (targeted): 1,133
Campaign 18 — control (untargeted): 1,336
Total households: 2,469


## 5. Takeaways

- **2,469 households**, **53 weeks** (1–53), **$4.60M** total transaction sales value — matches the
  documented Complete Journey scope.
- **Demographics cover only 801 of 2,469 households (~32%)** — demographic features will need
  explicit missing-value handling and shouldn't be assumed available for every household.
- **Campaigns 8, 13, and 18 (Type A)** target **1,076 / 1,077 / 1,133 households** respectively,
  consistent with the "~1,100 targeted households" assumption driving the campaign choice for
  the propensity/uplift split. Campaign 27 is also Type A but much smaller (361) and isn't used
  as a main campaign.
- **Redemption rates for the Type A campaigns are 14.7% / 18.2% / 18.9%** (8 / 13 / 18) — higher
  than most Type B/C campaigns, which mostly sit under 10%.
- **Every coupon redemption matches a targeted household** (0 unmatched pairs out of 792) — no
  leakage from control households "redeeming," which confirms redemption is only a valid outcome
  for targeted households and the uplift outcome must be spend/purchase instead.
- **Campaign 18 treated/control split: 1,133 treated vs. 1,336 control** households out of 2,469
  total — a reasonably balanced split for the out-of-time propensity test and uplift evaluation.
